In [2]:
import kagglehub
import pandas as pd
import os
from google.colab import userdata


# os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
# os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# # 1. Download latest version of dataset directly from Kaggle
# dataset_path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")
# print("Dataset downloaded at:", dataset_path)

# There was some problem in downloading via code so I downloaded diabetes.csv directly

# 2. Load the CSV file into a Pandas DataFrame
csv_file = os.path.join("../data/diabetes.csv")
df = pd.read_csv(csv_file)

# Display first 5 rows
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
# 1. Shape of dataset (Rows, Columns)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

# 2. Check data types and missing (null) values
print("--- Data Types & Null Values ---")
print(df.info())

# 3. Statistical summary of features
print("\n--- Statistical Summary ---")
display(df.describe().round(2))

# 4. Target Variable Distribution (0 = No Diabetes, 1 = Diabetes)
print("\n--- Target Class Distribution (Outcome) ---")
print(df["Outcome"].value_counts())

Dataset Shape: 768 rows, 9 columns

--- Data Types & Null Values ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None

--- Statistical Summary ---


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.00,768.00,768.00,768.00,768.00,768.00,768.00,768.00,768.00
mean,3.85,120.89,69.11,20.54,79.80,31.99,0.47,33.24,0.35
std,3.37,31.97,19.36,15.95,115.24,7.88,0.33,11.76,0.48
min,0.00,0.00,0.00,0.00,0.00,0.00,0.08,21.00,0.00
25%,1.00,99.00,62.00,0.00,0.00,27.30,0.24,24.00,0.00
50%,3.00,117.00,72.00,23.00,30.50,32.00,0.37,29.00,0.00
75%,6.00,140.25,80.00,32.00,127.25,36.60,0.63,41.00,1.00
max,17.00,199.00,122.00,99.00,846.00,67.10,2.42,81.00,1.00



--- Target Class Distribution (Outcome) ---
Outcome
0    500
1    268
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

# 1. Separate Input Features (X) and Target Column (y)
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

# 2. Split into 80% Train and 20% Test
# stratify=y ensures both train and test sets have the exact same ratio of Diabetic (1) vs Healthy (0) cases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training data shape : {X_train.shape}")
print(f"Testing data shape  : {X_test.shape}")

Training data shape : (614, 8)
Testing data shape  : (154, 8)


In [7]:
from sklearn.ensemble import RandomForestClassifier

# 1. Define model hyperparameters explicitly
model_params = {
    "n_estimators": 100,      # Number of decision trees in the forest
    "max_depth": 6,           # Restricts tree depth to prevent overfitting on 768 rows
    "min_samples_split": 5,   # Minimum samples required to split an internal node
    "random_state": 42        # Ensures reproducible results every time
}

# 2. Initialize and train the Random Forest model
model = RandomForestClassifier(**model_params)
model.fit(X_train, y_train)

print("Model trained successfully with parameters:")
print(model.get_params())

Model trained successfully with parameters:
{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 6, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 5, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# 1. Predict on test dataset
y_pred = model.predict(X_test)

# 2. Calculate Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%\n")

# 3. Confusion Matrix formatted as a readable table
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual: No Diabetes (0)", "Actual: Diabetes (1)"],
    columns=["Predicted: No Diabetes (0)", "Predicted: Diabetes (1)"]
)
print("--- Confusion Matrix ---")
display(cm_df)

# 4. Detailed Classification Report (Precision, Recall, F1-Score)
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["No Diabetes (0)", "Diabetes (1)"]))

Test Accuracy: 74.03%

--- Confusion Matrix ---


,Predicted: No Diabetes (0),Predicted: Diabetes (1)
Actual: No Diabetes (0),85,15
Actual: Diabetes (1),25,29



--- Classification Report ---
                 precision    recall  f1-score   support

No Diabetes (0)       0.77      0.85      0.81       100
   Diabetes (1)       0.66      0.54      0.59        54

       accuracy                           0.74       154
      macro avg       0.72      0.69      0.70       154
   weighted avg       0.73      0.74      0.73       154



In [9]:
# Sample input: [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age]
sample_patient = pd.DataFrame(
    [[2, 148, 72, 35, 0, 33.6, 0.627, 50]],
    columns=X.columns
)

prediction = model.predict(sample_patient)[0]
probability = model.predict_proba(sample_patient)[0][1]

print(f"Predicted Class : {prediction} ({'Diabetic' if prediction == 1 else 'Non-Diabetic'})")
print(f"Confidence/Risk : {probability * 100:.2f}%")

Predicted Class : 1 (Diabetic)
Confidence/Risk : 62.04%


In [10]:
import joblib
import sklearn
import os

# 1. Create a structured artifact containing the model, its parameters, and metadata
model_artifact = {
    "model": model,
    "params": model_params,                  # Explicitly defined hyperparameters
    "all_sklearn_params": model.get_params(), # Full internal parameter dictionary
    "feature_names": list(X.columns),        # Exact feature order for inference
    "test_accuracy": round(float(accuracy), 4),
    "sklearn_version": sklearn.__version__
}

# 2. Save the artifact to disk as model.joblib
joblib_filename = "model.joblib"
joblib.dump(model_artifact, joblib_filename)

# 3. Verify file creation and size
file_size_kb = os.path.getsize(joblib_filename) / 1024
print(f"Saved successfully : {joblib_filename} ({file_size_kb:.2f} KB)")
print(f"Scikit-Learn Ver.  : {sklearn.__version__}")
print(f"Joblib Version     : {joblib.__version__}")

Saved successfully : model.joblib (562.24 KB)
Scikit-Learn Ver.  : 1.6.1
Joblib Version     : 1.6.0


In [11]:
# 1. Load the saved artifact from disk
loaded_artifact = joblib.load("model.joblib")

loaded_model = loaded_artifact["model"]
loaded_params = loaded_artifact["params"]
loaded_features = loaded_artifact["feature_names"]

print("--- Loaded Model Parameters ---")
for k, v in loaded_params.items():
    print(f"{k}: {v}")

print(f"\nSaved Test Accuracy : {loaded_artifact['test_accuracy'] * 100:.2f}%")
print(f"Expected Features   : {loaded_features}")

# 2. Test prediction using the reloaded model
sample_input = pd.DataFrame(
    [[2, 148, 72, 35, 0, 33.6, 0.627, 50]],
    columns=loaded_features
)
test_pred = loaded_model.predict(sample_input)[0]
print(f"\nReloaded Model Sanity Prediction: {test_pred} ({'Diabetic' if test_pred == 1 else 'Non-Diabetic'})")

--- Loaded Model Parameters ---
n_estimators: 100
max_depth: 6
min_samples_split: 5
random_state: 42

Saved Test Accuracy : 74.03%
Expected Features   : ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

Reloaded Model Sanity Prediction: 1 (Diabetic)


In [12]:
from google.colab import files

# Trigger browser download for model.joblib
files.download("model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>